<a href="https://colab.research.google.com/github/ksuplee/tensorflow-nlp-tutorial/blob/main/14_NLP_Applications/14_02_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 14_02 NLP 시스템 평가 : 지표 계산과 한계 체험

**학습 목표**
- 생성 태스크 지표 **ROUGE·BLEU** 와 QA 지표 **EM·F1** 을 직접 계산해 본다.
- 표면(n-gram) 지표의 한계를 **의미 유사도(BERTScore 계열)** 와 비교해 확인한다.
- **LLM-as-a-judge** 개념과 간단한 **독성 필터** 를 체험한다.

> ※ 일부 셀은 사전학습 모델을 내려받아 실행합니다(첫 실행이 느릴 수 있음).

In [1]:
!pip install -q evaluate rouge_score sacrebleu sentence-transformers

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.6 MB/s eta 0:00:00


## 1. 생성 태스크 지표 — ROUGE·BLEU

요약·번역 평가에 쓰는 **표면 겹침(n-gram)** 기반 지표입니다. 의미가 같아도 **표현이 다르면 낮게** 나오는 한계를 확인합니다.

In [2]:
import evaluate
rouge = evaluate.load('rouge')
bleu = evaluate.load('sacrebleu')

pred = ['대규모 언어모델로 자연어처리 응용이 크게 확대되었다.']
ref  = ['LLM의 등장으로 NLP 활용 범위가 넓어졌다.']

r = rouge.compute(predictions=pred, references=ref)
b = bleu.compute(predictions=pred, references=[[ref[0]]])
print('ROUGE-L:', round(r['rougeL'], 3))
print('BLEU   :', round(b['score'], 1))
print('→ 의미는 비슷하지만 표현이 달라 표면 지표는 낮게 나온다.')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


ROUGE-L: 0.0
BLEU   : 6.6
→ 의미는 비슷하지만 표현이 달라 표면 지표는 낮게 나온다.


## 2. QA 지표 — EM(Exact Match)·F1

추출형 QA에서 예측 답과 정답의 일치를 측정합니다. **EM** 은 완전 일치, **F1** 은 부분 겹침(문자 단위)을 반영합니다.

In [3]:
from collections import Counter

def normalize(s):
    return s.strip().replace(' ', '')

def exact_match(pred, gold):
    return int(normalize(pred) == normalize(gold))

def f1_score(pred, gold):
    p, g = list(normalize(pred)), list(normalize(gold))  # 문자 단위 토큰
    common = sum((Counter(p) & Counter(g)).values())
    if common == 0:
        return 0.0
    prec, rec = common / len(p), common / len(g)
    return 2 * prec * rec / (prec + rec)

for pred, gold in [('2017년', '2017년'), ('2017년에', '2017년'), ('2018년', '2017년')]:
    print(f"pred={pred!r:9s} gold={gold!r:8s} -> EM={exact_match(pred, gold)}  F1={f1_score(pred, gold):.2f}")

pred='2017년'   gold='2017년'  -> EM=1  F1=1.00
pred='2017년에'  gold='2017년'  -> EM=0  F1=0.91
pred='2018년'   gold='2017년'  -> EM=0  F1=0.80


## 3. 의미 유사도 — BERTScore 계열

임베딩의 **의미적 유사도**로 평가하면, 표현이 달라도 의미가 같으면 높게 나옵니다. 여기서는 문장 임베딩의 **코사인 유사도**로 확인합니다(BERTScore의 핵심 아이디어).

In [ ]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('jhgan/ko-sroberta-multitask')
emb = model.encode([pred[0] if isinstance(pred, list) else pred,
                    ref[0] if isinstance(ref, list) else ref],
                   convert_to_tensor=True)
sim = util.cos_sim(emb[0], emb[1]).item()
print('두 문장:')
print(' 1)', 'LLM으로 NLP 응용이 크게 확대되었다.')
print(' 2)', 'LLM의 등장으로 NLP 활용 범위가 넓어졌다.')
print('의미 유사도(코사인):', round(sim, 3))
print('→ ROUGE/BLEU는 낮았지만 의미 유사도는 높다.')

## 4. LLM-as-a-judge (선택 · API 키 필요)

강력한 LLM에게 다른 모델의 답변을 채점시키는 방법입니다. 빠르고 저렴하지만 **심판 모델의 편향**에 유의해야 합니다.

In [ ]:
import os

judge_prompt = (
    "다음 답변을 1~5점으로 평가하라. 기준: 정확성·완결성.\n"
    "[질문] {q}\n"
    "[모범답안] {gold}\n"
    "[평가대상 답변] {pred}\n"
    '출력(JSON): {{"score": <1-5>, "reason": "..."}}'
)

q, gold, pred_ans = '트랜스포머는 언제 발표되었나?', '2017년', '2017년에 구글이 발표했다.'
filled = judge_prompt.format(q=q, gold=gold, pred=pred_ans)

if os.environ.get('OPENAI_API_KEY'):
    from openai import OpenAI
    client = OpenAI()
    resp = client.chat.completions.create(
        model='gpt-4o-mini', messages=[{'role': 'user', 'content': filled}])
    print(resp.choices[0].message.content)
else:
    print('OPENAI_API_KEY 없음 — 심판 프롬프트만 출력합니다.')
    print(filled)
    print('예상 출력: {"score": 5, "reason": "핵심 연도가 정확함"}')

## 5. 사회적 영향 — 간단한 독성 필터(개념)

모델 출력의 **독성·편향**은 사회적 영향을 낳습니다. 실무는 전용 모델·API를 쓰지만, 여기서는 원리를 규칙 기반으로 체험합니다.

In [ ]:
TOXIC_TERMS = ['바보', '멍청이', '쓸모없']

def toxicity_flag(text):
    hits = [w for w in TOXIC_TERMS if w in text]
    return {'flagged': bool(hits), 'terms': hits}

for t in ['정말 유용한 답변이네요.', '이 멍청이 같은 시스템']:
    print(f'{t}  ->  {toxicity_flag(t)}')

## 6. 정리

| 지표/방법 | 무엇을 보나 | 한계 |
|-----------|-------------|------|
| ROUGE·BLEU | n-gram 표면 겹침 | 표현 다르면 과소평가 |
| EM·F1 | 정답과의 일치/부분겹침 | 표면적 일치만 |
| 의미 유사도(BERTScore) | 임베딩 의미 유사도 | 계산비용↑ |
| LLM-as-a-judge | LLM이 채점 | 심판 편향 |
| 독성 필터 | 유해 표현 탐지 | 규칙의 한계 |

> 💡 지표는 참고일 뿐, 중요한 응용에서는 **사람 평가**와 **데이터 오염** 점검이 함께 필요합니다.